A memory-search model exposes study and retrieval operations that losses and
simulators can use. The model determines how its internal state changes and
how that state supports possible responses. `MemorySearch` describes the
shared interface, while a **model factory** connects an experiment and its
parameters to fresh model instances.

## The model interface

A compatible model supplies the attributes and methods in `MemorySearch`.
The selected declaration below shows the event lifecycle and probabilities;
the complete protocol also exposes item eligibility and retrieval activations.

Selected declaration from [`jaxcmr.typing`](https://github.com/githubpsyche/jaxcmr/blob/main/jaxcmr/typing.py):

```python
class MemorySearch(Protocol):
    item_count: int
    is_active: Bool[Array, ""]
    studied: Bool[Array, " item_count"]
    recall_total: Integer[Array, ""]
    study_index: Integer[Array, ""]

    def experience(self, choice: Int_) -> "MemorySearch": ...

    def start_retrieving(self) -> "MemorySearch": ...

    def retrieve(self, choice: Int_) -> "MemorySearch": ...

    def outcome_probability(self, choice: Int_) -> Float[Array, ""]: ...

    def outcome_probabilities(self) -> Float[Array, " recall_outcomes"]: ...
```

| Operation | Meaning |
|---|---|
| `experience(choice)` | Process a one-based within-list item code; zero is ignored. |
| `start_retrieving()` | Make the transition from study to retrieval. |
| `outcome_probabilities()` | Return the next-response distribution, with stop at index zero and item codes at their positive indices. |
| `retrieve(choice)` | Update after a supplied item or stop event; zero stops retrieval. |

State-changing methods return the updated model. Callers retain that return
value rather than assuming the original object was mutated. Parameters such
as context drift and learning rates configure the calculations; the changing
state includes context, learned associations and recall eligibility.

A [likelihood](losses.ipynb) supplies the recorded events, obtains their
probabilities and advances the model. A [simulator](simulation.ipynb)
samples events and advances the model. Both can consume a compatible model
without depending on its particular equations. JAX transformations also
require the implementation's state and operations to work with JAX.

## Factories connect models to experiments

Declaration from [`jaxcmr.typing`](https://github.com/githubpsyche/jaxcmr/blob/main/jaxcmr/typing.py):

```python
class MemorySearchModelFactory(Protocol):
    def __init__(
        self,
        dataset: RecallDataset,
        features: Float[Array, " word_pool_items features_count"] | None,
    ) -> None: ...

    def create_model(
        self, parameters: Mapping[str, Float_]
    ) -> MemorySearch: ...

    def create_trial_model(
        self, trial_index: Integer[Array, ""], parameters: Mapping[str, Float_]
    ) -> MemorySearch: ...
```

A factory **class** is passed as `model_factory_cls` to a loss or simulator.
Its constructor receives the dataset and optional vocabulary features.
The resulting factory **instance** creates fresh models from parameters.
`create_trial_model()` additionally receives a dataset row index, allowing
trial-specific stimulus features or conditions to affect construction.

For ordinary CMR, `jaxcmr.models.cmr.make_factory()` binds component creators
and returns this factory class. Its models use the dataset's maximum list
length; the ordinary factory does not use trial-specific features. Other
factories can require fields such as `pres_itemids` or `valence`.

This is the model entry point for existing workflows: supply a compatible
factory through `model_factory_cls`. Matching method names alone does not
establish suitable event coding or scientific assumptions.

In [ ]:
#| code-summary: "0 — Locate the project inputs"

from pathlib import Path

from jaxcmr.helpers import find_project_root, resolve_project_path

try:
    root_path = Path(find_project_root())
except FileNotFoundError:
    root_path = Path.cwd()


In [1]:
#| code-summary: "Create a model through a CMR factory"

from pathlib import Path
import json
import jax
import jax.numpy as jnp
from jaxcmr.helpers import load_data, generate_trial_mask
from jaxcmr.components import context, linear_memory
from jaxcmr.components.termination import NoStopTermination
from jaxcmr.models.cmr import make_factory

jax.config.update("jax_enable_x64", True)
data = load_data(resolve_project_path("data/Dupertuys2026.h5", root_path))
record = json.loads(Path(resolve_project_path("data/fits/Dupertuys2026_CMR_demo.json", root_path)).read_text())
parameters = {name: values[0] for name, values in record["fits"].items()
              if name != "subject"}
for name in ("allow_repeated_recalls", "learn_after_context_update"):
    parameters[name] = bool(parameters[name])

CMRFactory = make_factory(
    mfc_create_fn=linear_memory.init_mfc,
    mcf_create_fn=linear_memory.init_mcf,
    context_create_fn=context.init,
    termination_policy_create_fn=NoStopTermination,
)

factory = CMRFactory(data, features=None)
mixed = generate_trial_mask(data, "data['list_type'] == 1")
trial_index = int(jnp.flatnonzero(mixed)[0])
initial = factory.create_trial_model(trial_index, parameters)
print("Model:", type(initial).__name__)
print("Item capacity:", initial.item_count)
print("Initial study count:", int(initial.study_index))

Model: CMR
Item capacity: 12
Initial study count: 0


The factory configuration above selects CMR with linear associative
memories, temporal context and a no-stop policy. Parameter values come from
the supplied demonstration fit, but a model can equally use hand-chosen
values. [Model Components](components.ipynb) explains the creator functions.

## Following one event through the interface

Study events already contain within-list item codes. Recorded recalls
contain study positions, so the caller translates the first response
through `pres_itemnos` before passing it to `retrieve()`.

In [2]:
#| code-summary: "Inspect state and probabilities across one retrieval"

studied = initial
for item in data["pres_itemnos"][trial_index]:
    studied = studied.experience(item)
ready = studied.start_retrieving()
position = int(data["recalls"][trial_index, 0])
first_item = int(data["pres_itemnos"][trial_index, position - 1])
after = ready.retrieve(first_item)

print("Study counts, original and updated:",
      int(initial.study_index), int(studied.study_index))
print("Recall count after retrieval:", int(after.recall_total))
print("Probability of this item before retrieval:",
      float(ready.outcome_probability(first_item)))
print("Probability of this item after retrieval:",
      float(after.outcome_probability(first_item)))
print("Next-response vector shape:", after.outcome_probabilities().shape)

Study counts, original and updated: 0 12
Recall count after retrieval: 1
Probability of this item before retrieval: 0.13703638099013163
Probability of this item after retrieval: 0.0


Next-response vector shape: (13,)


The original model remains unstudied. The updated model has processed a
complete list and one response. With repeated recalls disabled, the recalled
item is no longer eligible; the remaining response distribution reflects
the changed state. A no-stop policy contributes no voluntary stopping
mechanism while candidates remain available.

The [fitting template](../templates/fitting.ipynb) uses this factory
configuration in a complete procedure. This page's event illustration
establishes the interface used by that procedure, not a test of CMR theory.